In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
import ce # type: ignore
import linear # type: ignore
import softmax # type: ignore
import relu # type: ignore
from common import repeat, test_checkgrad_1, test_compare_2, test_model_sgd_1, OneHot # type: ignore
from approx import approx # type: ignore

In [ ]:
class Per_Lin_ReLU_Lin_Softmax_CE_Autograd(nn.Module):
    """The version relying fully on PyTorch to handle `forward` and `backward` passes."""

    def __init__(self, in_features: int, hidden_features: int, out_features: int, w1 = None, b1 = None, w2 = None, b2 = None):
        super().__init__()

        self.linear1 = nn.Linear(in_features, hidden_features)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(hidden_features, out_features)
        self.loss_ = nn.CrossEntropyLoss(reduction='mean')

        if w1 is not None:
            with tr.no_grad():
                self.linear1.weight.copy_(w1)

        if b1 is not None:
            with tr.no_grad():
                self.linear1.bias.copy_(b1)

        if w2 is not None:
            with tr.no_grad():
                self.linear2.weight.copy_(w2)

        if b2 is not None:
            with tr.no_grad():
                self.linear2.bias.copy_(b2)

    def forward(self, x):
        return self.model(x)

    def model(self, x):
        z1 = self.linear1(x)
        a1 = self.relu(z1)
        return self.linear2(a1)

    def loss(self, p, t_labels):
        classes = self.linear2.weight.shape[0]
        t_onehot = OneHot.from_labels(classes)(t_labels.squeeze().long()).float()
        return self.loss_(p, t_onehot)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)

In [ ]:
class Per_Lin_ReLU_Lin_Softmax_CE_Backward(nn.Module):
    """
    The version where the `forward` and `backward` passes for each operation are written manually, 
    but PyTorch's autograd still orchestrates the overall gradient flow through the computational graph.
    """

    def __init__(self, in_features: int, hidden_features: int, out_features: int, w1 = None, b1 = None, w2 = None, b2 = None):
        super().__init__()

        self.linear1 = linear.Linear(in_features, hidden_features, w1, b1)
        self.relu = relu.ReLU()
        self.linear2 = linear.Linear(hidden_features, out_features, w2, b2)
        self.softmax = softmax.Softmax()
        self.loss_ = ce.CE(reduction='mean')

    def forward(self, x):
        return self.model(x)

    def model(self, x):
        z1 = self.linear1(x)
        a1 = self.relu(z1)
        return self.linear2(a1)

    def loss(self, p, t_labels):
        classes = self.linear2.weight.shape[0]
        t_onehot = OneHot.from_labels(classes)(t_labels.squeeze().long()).float()
        return self.loss_(self.softmax(p), t_onehot)

    def predict(self, x):
        with tr.no_grad():
            return self.model(x).argmax(dim=1)

In [ ]:
def test_logical_fn(model, bool_fn, samples=100, epochs=500, lr=0.1):
    x = (tr.randn(samples, 2, dtype=tr.float32) > 0).float()
    t = bool_fn(x[:, 0], x[:, 1]).long().unsqueeze(1)
    return test_model_sgd_1(model, x, t, epochs, lr=lr)


if __name__ == "__main__":
    def toOnehot(z: tr.Tensor) -> tr.Tensor:
        return OneHot.from_logits(z.shape[1])(z)

    def toLabel(z: tr.Tensor) -> tr.Tensor:
        return z.argmax(dim=1).float()

    test_compare_2(Per_Lin_ReLU_Lin_Softmax_CE_Autograd, Per_Lin_ReLU_Lin_Softmax_CE_Backward, 2, 3, 4, 5, E=3, fY=toLabel)
    test_compare_2(Per_Lin_ReLU_Lin_Softmax_CE_Autograd, Per_Lin_ReLU_Lin_Softmax_CE_Backward, 10, 20, 30, 40, E=3, fY=toLabel)

    for epochs in [500, 1000]:
        print(f"{epochs}/XOR: {repeat(lambda: test_logical_fn(Per_Lin_ReLU_Lin_Softmax_CE_Autograd(2, 4, 2), tr.logical_xor, epochs=epochs)):.2f}")